In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_json, expr, regexp_replace, split, size, slice,
    lower, trim, when, year, month, dayofweek, hour, length,
    udf, lit, concat_ws, substring_index, max as spark_max
)
from pyspark.sql.types import *

# ----------------------------------------------------------------------------------
# 1. Spark session + logging
# ----------------------------------------------------------------------------------
spark = (
    SparkSession.builder
        .appName("CleanGitHubCommits")
        .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
log = spark._jvm.org.apache.log4j.LogManager.getLogger("CleanGitHubCommits")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/11 15:45:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# ----------------------------------------------------------------------------------
# 2. Schema – only keep what we actually need
# ----------------------------------------------------------------------------------
file_schema = StructType([
    StructField("filename", StringType(), True),
    StructField("status", StringType(), True),
    StructField("additions", IntegerType(), True),
    StructField("deletions", IntegerType(), True)
])

commit_schema = StructType([
    StructField("sha", StringType(), True),
    StructField("commit", StructType([
        StructField("author", StructType([
            StructField("name", StringType(), True),
            StructField("email", StringType(), True),
            StructField("date", StringType(), True)
        ]), True),
        StructField("committer", StructType([
            StructField("name", StringType(), True),
            StructField("email", StringType(), True),
            StructField("date", StringType(), True)
        ]), True),
        StructField("message", StringType(), True)
    ]), True),
    StructField("html_url", StringType(), True),
    StructField("files", ArrayType(file_schema), True)
])

# ----------------------------------------------------------------------------------
# 3. Load raw Kafka messages
# ----------------------------------------------------------------------------------
df_raw = (
    spark.read
         .format("kafka")
         .option("kafka.bootstrap.servers", "kafka:29092")
         .option("subscribe", "github-commits")
         .option("startingOffsets", "earliest")
         .option("endingOffsets", "latest")
         .load()
)
count = df_raw.count()
print(f"Successfully loaded {count} records from topic 'github-commits'.")



[Stage 0:>                                                          (0 + 1) / 1]

Successfully loaded 121 records from topic 'github-commits'.


In [3]:

# Kafka’s `value` is binary; cast to string and parse JSON
# Step 3: Parse and drop malformed/null-key rows early
df_json = (
    df_raw
      .select(from_json(col("value").cast("string"), commit_schema).alias("c"))
      .select("c.*")
      .filter(
          col("commit").isNotNull() &
          col("commit.message").isNotNull() &
          col("files").isNotNull() &
          (size(col("files")) > 0) &
          col("sha").isNotNull() &
          col("html_url").isNotNull()
      )
)

# ----------------------------------------------------------------------------------
# 4. Helper UDFs / expressions
# ----------------------------------------------------------------------------------

# Simple language inference from filename extension
ext2lang = {
    "py": "Python", "java": "Java", "js": "JavaScript", "ts": "TypeScript",
    "cpp": "C++", "c": "C", "cs": "C#", "rb": "Ruby", "go": "Go",
    "rs": "Rust", "kt": "Kotlin", "swift": "Swift", "php": "PHP",
    "scala": "Scala", "md": "Markdown", "yml": "YAML", "yaml": "YAML",
}

# Change‑type heuristic
change_expr = (
    when(lower(col("commit_message_clean")).rlike(r"\bfix(e[ds])?\b|bug"), "fix")
    .when(lower(col("commit_message_clean")).rlike(r"\brefactor|clean"), "refactor")
    .when(lower(col("commit_message_clean")).rlike(r"\b(add|feature|implement)"), "feature")
    .otherwise("other")
)


@udf("string")
def infer_lang(filename):
    if not filename:
        return None
    ext = filename.rsplit('.', 1)[-1].lower() if '.' in filename else ""
    return ext2lang.get(ext, "Other")

def strip_bot(email):
    if email is None:
        return None
    e = email.lower()
    if "noreply" in e or "bot@" in e:
        return None
    return e

strip_bot_udf = udf(strip_bot, StringType())



In [4]:

# ----------------------------------------------------------------------------------
# 5. Flatten to file‑level rows
# ----------------------------------------------------------------------------------

df_exp = (
    df_json
      .withColumn("repo",             substring_index(col("html_url"), "/", -2))
      .withColumn("author_name_raw",  col("commit.author.name"))
      .withColumn("author_email_raw", col("commit.author.email"))
      .withColumn("committer_name",   col("commit.committer.name"))
      .withColumn("committer_email",  col("commit.committer.email"))
      .withColumn("commit_timestamp", col("commit.author.date").cast("timestamp"))
      .withColumn("commit_message_raw", col("commit.message"))
      .withColumn("files",            col("files"))
      .withColumn("author_name",      lower(trim(col("author_name_raw"))))
      .withColumn("author_email",     strip_bot_udf(col("author_email_raw")))
      .drop("author_name_raw", "author_email_raw")
      .withColumn("commit_message_clean",
                  lower(regexp_replace(col("commit_message_raw"), r"http\S+|\p{So}", "")))
      .withColumn("year",        year("commit_timestamp"))
      .withColumn("month",       month("commit_timestamp"))
      .withColumn("day_of_week", dayofweek("commit_timestamp") - 2)
      .withColumn("hour_of_day", hour("commit_timestamp"))
      .withColumn("file",        expr("explode(files)"))
      .selectExpr(
          "repo",
          "sha as commit_sha",
          "author_name",
          "author_email",
          "committer_name",
          "committer_email",
          "file.filename as file_path",
          "file.status as change_status",
          "file.additions as lines_added",
          "file.deletions as lines_deleted",
          "commit_message_clean",
          "commit_timestamp",
          "year","month","day_of_week","hour_of_day"
      )
      .filter(col("file_path").isNotNull())  # extra guard
)
df_exp = df_exp.filter(col("file_path").isNotNull() & (col("file_path") != ""))


In [5]:

# ----------------------------------------------------------------------------------
# 6. File‑level enrichments
# ----------------------------------------------------------------------------------
df_clean = (
    df_exp
      .withColumn("file_name", split(col("file_path"), "/").getItem(-1))
      .withColumn("path_depth", size(split(col("file_path"), "/")) - 1)
      .withColumn("module_candidate",
                  concat_ws("/",
                            slice(split(col("file_path"), "/"), 1, 2)))  # first 1‑2 folders
      .withColumn("language", infer_lang(col("file_name")))
      .withColumn("commit_size", col("lines_added") + col("lines_deleted"))
      .withColumn("change_type", change_expr)
      .dropDuplicates(["commit_sha", "file_path"])  # defensive
)

# ----------------------------------------------------------------------------------
# 7. Persist or hand off
# ----------------------------------------------------------------------------------
(df_clean
  .write
  .mode("overwrite")
  .parquet("output")
)

log.info(f"Written {df_clean.count()} clean file‑level rows")


25/07/11 15:45:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
25/07/11 15:45:44 WARN KafkaDataConsumer: KafkaDataConsumer is not running in UninterruptibleThread. It may hang when KafkaDataConsumer's methods are interrupted because of KAFKA-1894
25/07/11 15:45:44 WARN KafkaDataConsumer: KafkaDataConsumer is not running in UninterruptibleThread. It may hang when KafkaDataConsumer's methods are interrupted because of KAFKA-1894
25/07/11 15:45:44 WARN KafkaDataConsumer: KafkaDataConsumer is not running in UninterruptibleThread. It may hang when KafkaDataConsumer's methods are interrupted because of KAFKA-1894
25/07/11 15:45:44 WARN KafkaDataConsumer: KafkaDataConsumer is not running in UninterruptibleThread. It may hang when KafkaDataConsumer's methods are interrupted because of KAFKA-1894
25/07/11 15:45:44 WARN KafkaDataConsumer: KafkaDataConsumer is not running 

In [16]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from math import log2

# Map each (commit, module) pair
df_commit_module = (
    df_clean_filtered  # <- the version that keeps only folder-level modules
      .select("commit_sha", "module_candidate")
      .distinct()
)
# Count modules per commit and compute p_i = 1 / N  (uniform because we don't weight files)
w_commit = Window.partitionBy("commit_sha")

df_entropy_piece = (
    df_commit_module
      .withColumn("modules_in_commit", F.count("*").over(w_commit))
      .withColumn(
          "entropy_piece",
          F.when(F.col("modules_in_commit") == 1, F.lit(0.0))  # log2(1)=0 → entropy 0
           .otherwise(
               - (1 / F.col("modules_in_commit")) *
                 F.log2(1 / F.col("modules_in_commit"))
           )
      )
)


In [20]:
w_module = Window.partitionBy("module_candidate")

df_entropy_density = (
    df_entropy_piece
      .withColumn("entropy_sum", F.sum("entropy_piece").over(w_module))
      .withColumn("commit_cnt",  F.count("*").over(w_module))
      .withColumn(
          "density_numer",   # commits where module was the ONLY module
          F.sum(
              F.when(F.col("modules_in_commit") == 1, F.lit(1)).otherwise(F.lit(0))
          ).over(w_module)
      )
      .select(
          "module_candidate",
          "entropy_sum",
          "commit_cnt",
          "density_numer"
      )
      .distinct()
      .withColumn("avg_entropy", F.col("entropy_sum") / F.col("commit_cnt"))
      .withColumn("density",     F.col("density_numer") / F.col("commit_cnt"))
      .drop("entropy_sum", "density_numer")
)
df_entropy_density = df_entropy_density.withColumn(
    "norm_entropy",
    F.when(F.col("avg_entropy") == 0, F.lit(0.0))
     .otherwise(F.col("avg_entropy") / F.lit(log2( df_commit_module.select("module_candidate").distinct().count() )))
)
from pyspark.sql.functions import lit

from pyspark.sql.functions import unix_timestamp, lit, col

df_modules = (
    df_module_base
      # Compute age in days
      .withColumn("age_days", (
          unix_timestamp("last_commit_time") - unix_timestamp("first_commit_time")
      ) / 86400)

      # Add scores and flags
      .withColumn("single_author_module", (col("num_contributors") == 1))
      .withColumn("popularity_score", 0.5 * col("num_contributors") + 0.5 * col("num_commits"))
      .withColumn("stability_score", 1 / (1 + col("total_churn")))
      .withColumn("maturity_score", col("age_days") / (col("age_days") + 30))  # normalized age

      # Join entropy + density for cohesion_score
      .join(df_entropy_density, on="module_candidate", how="left")
      .withColumn("cohesion_score",
                  (1 - col("norm_entropy")) * lit(0.7) + col("density") * lit(0.3))

      # Clean up intermediate columns
      .drop("avg_entropy", "norm_entropy", "density")
)
pd_summary = df_modules.describe().toPandas().set_index("summary").round(2)



25/07/11 15:58:45 WARN KafkaDataConsumer: KafkaDataConsumer is not running in UninterruptibleThread. It may hang when KafkaDataConsumer's methods are interrupted because of KAFKA-1894
25/07/11 15:58:45 WARN KafkaDataConsumer: KafkaDataConsumer is not running in UninterruptibleThread. It may hang when KafkaDataConsumer's methods are interrupted because of KAFKA-1894
25/07/11 15:58:45 WARN KafkaDataConsumer: KafkaDataConsumer is not running in UninterruptibleThread. It may hang when KafkaDataConsumer's methods are interrupted because of KAFKA-1894
25/07/11 15:58:45 WARN KafkaDataConsumer: KafkaDataConsumer is not running in UninterruptibleThread. It may hang when KafkaDataConsumer's methods are interrupted because of KAFKA-1894
25/07/11 15:58:45 WARN KafkaDataConsumer: KafkaDataConsumer is not running in UninterruptibleThread. It may hang when KafkaDataConsumer's methods are interrupted because of KAFKA-1894
25/07/11 15:58:45 WARN KafkaDataConsumer: KafkaDataConsumer is not running in Un

        module_candidate    num_contributors         num_commits  \
summary                                                            
count                 66                  66                  66   
mean                None  1.2727272727272727  3.5757575757575757   
stddev              None  0.7135060680126758    3.88314149610697   
min        .cursor/rules                   0                   1   
max          test/system                   4                  25   

            avg_commit_size        total_churn            age_days  \
summary                                                              
count                    66                 66                  66   
mean     50.080238798328786  440.1060606060606   1.757692024410775   
stddev   60.874335833850076   826.183498499863  3.2185530433873795   
min                     2.0                  4                 0.0   
max                   337.0               4412  13.302233796296296   

           popularity_score     

In [21]:
print(pd_summary)

        module_candidate    num_contributors         num_commits  \
summary                                                            
count                 66                  66                  66   
mean                None  1.2727272727272727  3.5757575757575757   
stddev              None  0.7135060680126758    3.88314149610697   
min        .cursor/rules                   0                   1   
max          test/system                   4                  25   

            avg_commit_size        total_churn            age_days  \
summary                                                              
count                    66                 66                  66   
mean     50.080238798328786  440.1060606060606   1.757692024410775   
stddev   60.874335833850076   826.183498499863  3.2185530433873795   
min                     2.0                  4                 0.0   
max                   337.0               4412  13.302233796296296   

           popularity_score     